Paper: https://aclanthology.org/2021.emnlp-main.612.pdf

Github: https://github.com/nattaptiy/qe_disentangled

Eval data of paper: 
- Task 3 Document-Level QA: https://github.com/facebookresearch/mlqe, https://www.statmt.org/wmt20/quality-estimation-task.html
- STS17: https://public.ukp.informatik.tu-darmstadt.de/reimers/sentence-transformers/datasets/STS2017-extended.zip

# Encoder

In [1]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('sentence-transformers/LaBSE')

sentence = 'Find embedding size'
embedding_dim = encoder.encode(sentence).shape[0]
print(embedding_dim)

768


In [2]:
# import gc
# import torch

# def free(encoder: SentenceTransformer):
#     try:
#         encoder.to('cpu')
#     except:
#         pass

#     del encoder

#     gc.collect()

#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         # torch.cuda.synchronize()   # optional
#         gc.collect()
#         torch.cuda.empty_cache()

# Train, Val Split

In [3]:
# # Run ONCE
# import pandas as pd
# import glob
# import os

# VAL_SIZE = 0.1
# PATH = '../data/Tatoeba'
# TRAIN_SUFFIX = '_Train'
# VAL_SUFFIX   = '_Val'

# os.makedirs(f"{PATH}{VAL_SUFFIX}", exist_ok=True)
# os.makedirs(f"{PATH}{TRAIN_SUFFIX}", exist_ok=True)

# all_tsv = glob.glob(f'{PATH}/*.tsv')

# for tsv_path in all_tsv:
#     filename = os.path.basename(tsv_path)
#     name_only = os.path.splitext(filename)[0]

#     try:
#         df = pd.read_csv(
#             tsv_path,
#             sep='\t',
#             header=None,
#             names=['src_id', 'src', 'tar_id', 'tar'],
#             on_bad_lines='skip',      
#             dtype=str
#         )
        
#         if len(df) == 0:
#             print(f"Empty file: {tsv_path}")
#             continue
            
#         # Shuffle và split
#         val_df = df.sample(frac=VAL_SIZE, random_state=42)          # fixed seed → reproducible
#         train_df = df.drop(val_df.index)                            # cách an toàn nhất
        
#         # Save to folders
#         val_path  = f"{PATH}{VAL_SUFFIX}/{filename}"
#         train_path = f"{PATH}{TRAIN_SUFFIX}/{filename}"

#         val_df.to_csv(val_path, sep='\t', header=False, index=False)
#         train_df.to_csv(train_path, sep='\t', header=False, index=False)
        
#         print(f"Done | train: {len(train_df):,} | val: {len(val_df):,}")
        
#     except Exception as e:
#         print(f"Error Processing {tsv_path}: {str(e)}")
#         continue

# Dataset

In [4]:
from dataset import TatoebaDataset

#train_dataset = TatoebaDataset('../data/Tatoeba_Train', encoder=encoder)
val_dataset = TatoebaDataset('../data/Tatoeba_Val', encoder=encoder)

In [5]:
#free(encoder=encoder)

In [6]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

#train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE)
train_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE)

# Model

In [7]:
from model import DREAMModel

model = DREAMModel(embedding_dim, num_languages=8)

# Optimizer

In [8]:
import torch

LEARNING_RATE = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Loss

In [9]:
import torch
import torch.nn.functional as F

def reconstruction_loss(e, eM, eL):
    return ((e - (eM + eL)) ** 2).mean()

def meaning_loss(sM, tM, rand_sM, rand_tM):
    # The meaning of 2 parallel sentences should be close
    parallel_sim = F.cosine_similarity(sM, tM, dim=-1)
    Lx = (1 - parallel_sim).mean()

    # The meaning of 2 random sentences should be far
    source_sim = F.cosine_similarity(sM, rand_sM, dim=-1)
    random_sim = F.cosine_similarity(tM, rand_tM)
    Lm = (torch.clamp(source_sim, min=0) + torch.clamp(random_sim, min=0)).mean()

    return Lx + Lm

def language_loss(sL, tL, rand_sL, rand_tL, sI, tI, src_lang_id, tar_lang_id):
    # Same-language embeddings should be close (được optimize)
    Lm = (2 - F.cosine_similarity(sL, rand_sL, dim=-1) 
             - F.cosine_similarity(tL, rand_tL, dim=-1)).mean()

    # Cross-language embeddings should be far (for reserving the author's code only)
    Lm_cross = torch.clamp(F.cosine_similarity(sL, tL, dim=-1), min=0).mean()

    # Language identification
    Li = F.cross_entropy(sI, src_lang_id) + F.cross_entropy(tI, tar_lang_id)

    return Lm + Li


def total_loss(e_src, e_trg,
               eM_src, eM_trg, eM_rand_src, eM_rand_trg,
               eL_src, eL_trg, eL_rand_src, eL_rand_trg,
               eI_src, eI_trg,
               src_lang_id, tar_lang_id):

    LR = reconstruction_loss(e_src, eM_src, eL_src) \
       + reconstruction_loss(e_trg, eM_trg, eL_trg)

    LM = meaning_loss(eM_src, eM_trg, eM_rand_src, eM_rand_trg)

    LL = language_loss(eL_src, eL_trg, eL_rand_src, eL_rand_trg,
                       eI_src, eI_trg,
                       src_lang_id, tar_lang_id)

    return LR + LM + LL, LR, LM, LL

# Train

In [10]:
import time

EPOCH = 10

for epoch in range(1, EPOCH + 1):
    t0 = time.time()

    model.train()
    for batch in train_loader:
        e_src, e_trg, rand_src, rand_tar, src_lang_id, tar_lang_id = [x for x in batch]

        optimizer.zero_grad()
        print(e_src)

        eL_src, eM_src, eI_src = model(e_src)
        eL_trg, eM_trg, eI_trg = model(e_trg)
        eL_rand_src, eM_rand_src, _ = model(rand_src)
        eL_rand_trg, eM_rand_trg, _ = model(rand_tar)

        train_loss, loss_rec, loss_mean, loss_lang = total_loss(e_src, e_trg, eM_src, eM_trg,
                                                        eM_rand_src, eM_rand_trg, eL_src, eL_trg, eL_rand_src, eL_rand_trg, 
                                                        eI_src, eI_trg,src_lang_id, tar_lang_id)

        train_loss.backward()
        optimizer.step()


    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            e_src, e_trg, rand_src, rand_tar, src_lang_id, tar_lang_id = [x for x in batch]

            eL_src, eM_src, eI_src = model(e_src)
            eL_trg, eM_trg, eI_trg = model(e_trg)
            eL_rand_src, eM_rand_src, _ = model(rand_src)
            eL_rand_trg, eM_rand_trg, _ = model(rand_tar)

            val_loss, loss_rec, loss_mean, loss_lang = total_loss(e_src, e_trg, eM_src, eM_trg,
                                                    eM_rand_src, eM_rand_trg, eL_src, eL_trg, eL_rand_src, eL_rand_trg, 
                                                    eI_src, eI_trg,src_lang_id, tar_lang_id)
    
    t1 = time.time()

    print(f"Epoch {epoch:3d} | train={train_loss:.4f} (LR={loss_rec:.3f} LM={loss_mean:.3f} LL={loss_lang:.3f}) | val={val_loss:.4f}")

("This coffee is so hot that I can't drink it.", 'I borrowed this book from him.', 'Tom said he thought he could pass the test.', 'Our teacher made us work in groups.', 'He fired Mary.', "Tom doesn't like my dog.", "I think you're too drunk to drive.", "Tom can't go.", 'Not knowing what to say, I remained silent.', 'Dogs can\'t talk, but it\'s almost as if the puppy\'s eyes said, "No, I don\'t have a home."', "It's based on a true story.", 'He retched and then started puking.', 'You should have told it to me sooner.', "I'm sure it won't be easy to do that.", 'Tom appears to be winning.', 'My children are hungry.', "Tom didn't like that.", "I'm trying hard to make this feel like a compliment.", "I'll leave this work to you.", 'This technology will open up a whole new avenue of research.', "I haven't eaten anything since yesterday.", "Don't put your fingers in the meat grinder!", 'Nobody has ever done this before.', 'You are improving!', 'Tell him it was all your fault.', "You're not the

TypeError: linear(): argument 'input' (position 1) must be Tensor, not tuple